In [1]:
import json
from pathlib import Path
import pandas as pd
from form990_parser import Form990Parser

%load_ext autoreload
%autoreload 2

In [2]:
file_lookup = pd.read_csv('example_usecase_lookup.csv')
file_lookup.sample(5)

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
1073,A 501c3 filed a 990PF containing AllOthProgRlt...,922613857,Oak City Locksport Inc,501c3,xml\2026\2026_3A\202620639349100002_public.xml,2025,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
943,A 501c3 filed a 990PF containing InvestmentsCo...,461463656,ELEVEN ELEVEN FOUNDATION,501c3,xml\2025\2025_11C\202523219349108507_public.xml,2024,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
562,A 501c3 filed a 990 containing IRS990ScheduleO...,752721254,HISPANIC CULTURAL CENTER OF MIDLAND,501c3,xml\2023\2023_01A\202300199349300600_public.xml,2021,990,1.0,222.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
328,A 501c3 filed a 990 containing IRS990ScheduleN...,452641647,SUNCOAST AQUATIC NATURE CENTER,501c3,xml\2021\2021_01A\202122289349300712_public.xml,2019,990,1.0,228.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1118,A 501c4 filed a 990EZ containing IRS990Schedul...,931451174,CITIZENS ORGANIZED TO RETHINK EXPANSION OF 522,501c4,xml\2026\2026_2A\202640429349200209_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0


In [29]:
schedules = [
    'IRS990ScheduleC',
    'IRS990ScheduleI'
]

df = file_lookup.loc[
    (file_lookup.loc[:, schedules[0]] == 1) | (file_lookup.loc[:, schedules[1]] == 1)
]

tax_years = [
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]

df = df.loc[df.loc[:, 'TaxYr'].isin(tax_years)]

return_types = [
    # '990EZ',
    '990',
    # '990PF',
    # '990-T'
]

if len(return_types) < 4:
    df = df.loc[df.loc[:, 'ReturnTypeCd'].isin(return_types)]
else:
    pass

org_types = [
    '501c3',
    '501c4',
    '527'
]

if len(org_types) < 3:
    df = df.loc[df.loc[:, 'OrgType'].isin(org_types)]
else:
    pass

In [30]:
import time

In [34]:
header = pd.DataFrame()
returns = pd.DataFrame()
members = pd.DataFrame()
contractors = pd.DataFrame()

schedule_content_accum = {
    'IRS990ScheduleC': {
        'Section527PoliticalOrgGrp': pd.DataFrame(),
        'AvgLobbyingNontaxableAmountGrp': pd.DataFrame(),
        'AvgTotalLobbyingExpendGrp': pd.DataFrame(),
        'AvgGrassrootsNontaxableGrp': pd.DataFrame(),
        'AvgGrassrootsLobbyingExpendGrp': pd.DataFrame(),
        'SupplementalInformationDetail': pd.DataFrame(),
        'Flat Content': pd.DataFrame()
    },
    'IRS990ScheduleI': {
        'RecipientTable': pd.DataFrame(),
        'GrantsOtherAsstToIndivInUSGrp': pd.DataFrame(),
        'SupplementalInformationDetail': pd.DataFrame(),
        'Flat Content': pd.DataFrame()
    }
}

In [ ]:
total_parse_time = 0
flat_content = dict[tuple[int, str], str]
nested_content = dict[str, list[dict[str, str | dict]]]

def flat_content_formatting(_flat_content_key: tuple[int, str], _flat_content_val: str) -> tuple[str, str]:
    return _flat_content_key[1], _flat_content_val

def nested_content_formatting(_nested_content_dict: dict):
    return _nested_content_dict['Nested Content']

with open('IRS_Form_990_Generic_Formatting.json', mode='r') as f:
    gen_formatter = json.loads(f.read())['ReturnData']['Children']['IRS990']['Children']

for i, file in enumerate(df.file.unique()):
    example_file_path = Path(file)
    prev_time = time.time()



    parser = Form990Parser(example_file_path)

    full_formatter = parser.read_in_formatter('IRS_Form_990_Formatting.json')
    # * Header
    header_formatter = full_formatter['Header']
    curr_header = parser.parse_header(
        _header_hierarchy_dict = header_formatter
    )
    header = pd.concat([header, curr_header], axis=0, ignore_index=True)
    
    curr_ein = curr_header.loc[:, 'EIN'].values[0]
    curr_tax_year = curr_header.loc[:, 'TaxYear'].values[0]
    # curr_return_type = curr_header.loc[:, 'ReturnTypeCd'].values[0]
    # * Return
    return_nested_targets = parser.create_xml_target_mapping_list(
        _target_type=nested_content,
        _format_mappings_func=nested_content_formatting,
        _dict=full_formatter['Return']['990']
    )
    # print(file)
    return_data = parser.parse_return(
        _ein=curr_ein,
        _tax_year=curr_tax_year,
        _return_root=parser.find_element('IRS990'),
        _element_group_mappings=return_nested_targets,
        _return_parsing_format=gen_formatter
    )

    returns = pd.concat([returns, return_data['return_data']], axis=0, ignore_index=True)
    members = pd.concat([members, return_data['Form990PartVIISectionAGrp']], axis=0, ignore_index=True)
    contractors = pd.concat([contractors, return_data['ContractorCompensationGrp']], axis=0, ignore_index=True)
    for schedule in schedules:
        # No need to do parsing when our reference file has already checked if it's present
        if len(df.loc[(df.file == file) & (df.loc[:, schedule] == 1)]) == 0:
            continue

        schedule_formatter = full_formatter[schedule]['Content']

        schedule_flat_targets = parser.create_xml_target_mapping_dict(
            _target_type=flat_content,
            _format_mappings_func=flat_content_formatting,
            _dict=schedule_formatter
        )

        # Nested targets
        schedule_nested_targets = parser.create_xml_target_mapping_list(
            _target_type=nested_content,
            _format_mappings_func=nested_content_formatting,
            _dict=schedule_formatter
        )

        schedule_content, missed_content = parser.parse_schedule(
            _ein=curr_ein,
            _tax_year=curr_tax_year,
            _schedule_root=parser.find_element(schedule),
            _flat_targets=schedule_flat_targets,
            _nested_targets=schedule_nested_targets
        )

        for content_key, content_df in schedule_content.items():
            schedule_content_accum[schedule][content_key] = pd.concat(
                [
                    schedule_content_accum[schedule][content_key],
                    content_df
                ],
                axis=0,
                ignore_index=True
            )
        if len(missed_content) > 0:
            print(file)
            print("\t", schedule)
            print("\t", missed_content)



    final_time = time.time()
    total_parse_time += final_time - prev_time
total_parse_time / (i+1)

xml\2019\2019_03A\201913159349303926_public.xml
xml\2019\2019_03A\201913189349314251_public.xml
xml\2019\2019_03A\201913179349302811_public.xml
xml\2020\2020_02A\202010489349300101_public.xml
xml\2020\2020_03A\202011979349302981_public.xml
xml\2020\2020_02A\202010459349301146_public.xml
xml\2020\2020_04A\202021909349301007_public.xml
xml\2019\2019_02A\201903199349310630_public.xml
xml\2019\2019_02A\201903179349303965_public.xml
xml\2019\2019_02A\201903189349304325_public.xml
xml\2020\2020_04A\202021969349302832_public.xml
xml\2019\2019_03A\201912949349301406_public.xml
xml\2020\2020_03A\202011979349304051_public.xml
xml\2019\2019_02A\201903059349301130_public.xml
xml\2019\2019_02A\201903109349301005_public.xml
xml\2020\2020_02A\202011269349300206_public.xml
xml\2019\2019_03A\201913029349301131_public.xml
xml\2019\2019_02A\201903179349301005_public.xml
xml\2019\2019_02A\201903159349302410_public.xml
xml\2020\2020_04A\202021299349300927_public.xml
xml\2020\2020_03A\202011979349303991_pub

0.21978000374727472

In [39]:
header.to_csv('example_xmls_output/header.csv', index=False)
returns.to_csv('example_xmls_output/returns.csv', index=False)
members.to_csv('example_xmls_output/members.csv', index=False)
contractors.to_csv('example_xmls_output/contractors.csv', index=False)
for schedule, schedule_df_dict in schedule_content_accum.items():
    for schedule_df_name, schedule_df in schedule_df_dict.items():
        schedule_df.to_csv(f'example_xmls_output/{schedule}_{schedule_df_name}.csv', index=False)